# Program 11: Class-Weighted GA for cwk-NN (10-Fold CV)

This notebook implements the genetic algorithm setup and cross-validation workflow for class-weighted feature weighting with a simplified cwk-NN proxy.

In [1]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# --- GA Parameters based on the paper ---
POPULATION_SIZE = 21       # [cite: 66, 93]
CROSSOVER_RATE = 0.8       # pc = 0.8 [cite: 70, 94]
MUTATION_RATE = 0.01       # pm = 0.01 [cite: 71, 95]
MAX_GENERATIONS = 20       # [cite: 76]
NO_IMPROVEMENT_LIMIT = 10  # [cite: 76]

# --- Dataset Characteristics ---
NUM_CASES = 951            # [cite: 44]
NUM_ATTRIBUTES = 94        # [cite: 50]
NUM_CLASSES = 7            # [cite: 44]

In [2]:
def initialize_population(pop_size, num_classes, num_attributes):
    """
    Initializes the population with real-valued weights between 0 and 1.
    Each individual has 7 class-specific weight sets for the 94 attributes.
    In the paper, this was seeded with expert/ML weights, but here we use random initialization.
    """
    return np.random.rand(pop_size, num_classes, num_attributes)

def evaluate_fitness(individual, X_train, y_train, X_test, y_test):
    """
    Evaluates an individual's fitness.
    Note: The paper uses custom distance metrics like HVDM for cwk-NN.
    This implementation uses a weighted Euclidean stand-in.
    """
    mean_weights = np.mean(individual, axis=0)
    X_train_weighted = X_train * mean_weights
    X_test_weighted = X_test * mean_weights

    knn = KNeighborsClassifier(n_neighbors=7)
    knn.fit(X_train_weighted, y_train)
    predictions = knn.predict(X_test_weighted)
    return accuracy_score(y_test, predictions)

def roulette_wheel_selection(population, fitness_scores):
    """Select parents using fitness-proportionate selection."""
    total_fitness = np.sum(fitness_scores)
    if total_fitness == 0:
        probs = np.ones(len(population)) / len(population)
    else:
        probs = fitness_scores / total_fitness

    cumulative_probs = np.cumsum(probs)
    mating_pool = []
    while len(mating_pool) < POPULATION_SIZE:
        r = np.random.rand()
        for j, cum_prob in enumerate(cumulative_probs):
            if r <= cum_prob:
                mating_pool.append(population[j].copy())
                break
    return mating_pool

def uniform_crossover(parent1, parent2):
    """Uniform crossover with discrete recombination."""
    child1, child2 = parent1.copy(), parent2.copy()
    if np.random.rand() < CROSSOVER_RATE:
        mask = np.random.rand(NUM_CLASSES, NUM_ATTRIBUTES) > 0.5
        child1[mask] = parent2[mask]
        child2[mask] = parent1[mask]
    return child1, child2

def uniform_mutation(individual):
    """Uniform mutation replacing genes with random values in [0, 1]."""
    mask = np.random.rand(NUM_CLASSES, NUM_ATTRIBUTES) < MUTATION_RATE
    random_values = np.random.rand(NUM_CLASSES, NUM_ATTRIBUTES)
    individual[mask] = random_values[mask]
    return individual

In [3]:
def run_genetic_algorithm(X_train, y_train, X_test, y_test):
    """Main GA loop."""
    population = initialize_population(POPULATION_SIZE, NUM_CLASSES, NUM_ATTRIBUTES)

    best_overall_individual = None
    best_overall_fitness = -1.0
    generations_without_improvement = 0

    for generation in range(MAX_GENERATIONS):
        fitness_scores = np.array([
            evaluate_fitness(ind, X_train, y_train, X_test, y_test)
            for ind in population
        ])

        elite_idx = np.argmax(fitness_scores)
        elite_individual = population[elite_idx].copy()
        current_best_fitness = fitness_scores[elite_idx]

        if current_best_fitness > best_overall_fitness:
            best_overall_fitness = current_best_fitness
            best_overall_individual = elite_individual.copy()
            generations_without_improvement = 0
        else:
            generations_without_improvement += 1

        print(f"Gen {generation+1}: Best Accuracy = {current_best_fitness:.4f}")

        if generations_without_improvement >= NO_IMPROVEMENT_LIMIT:
            print("Stopping early: No improvement in 10 generations.")
            break

        mating_pool = roulette_wheel_selection(population, fitness_scores)

        new_population = []
        for i in range(0, len(mating_pool) - 1, 2):
            p1, p2 = mating_pool[i], mating_pool[i + 1]
            c1, c2 = uniform_crossover(p1, p2)
            new_population.extend([c1, c2])

        if len(new_population) < POPULATION_SIZE:
            new_population.append(mating_pool[-1].copy())

        new_population = [uniform_mutation(ind) for ind in new_population]
        new_population.append(elite_individual)

        if len(new_population) > POPULATION_SIZE:
            temp_fitness = np.array([
                evaluate_fitness(ind, X_train, y_train, X_test, y_test)
                for ind in new_population
            ])
            sorted_indices = np.argsort(temp_fitness)[::-1]
            population = [new_population[i] for i in sorted_indices[:POPULATION_SIZE]]
        else:
            population = new_population

    return best_overall_individual, best_overall_fitness

In [4]:
# --- Main Execution block to simulate the 10-fold CV ---
np.random.seed(42)
X = np.random.rand(NUM_CASES, NUM_ATTRIBUTES)
y = np.random.randint(0, NUM_CLASSES, NUM_CASES)

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
fold_accuracies = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- Starting CV Fold {fold+1} ---")
    X_train, y_train = X[train_idx], y[train_idx]
    X_test, y_test = X[test_idx], y[test_idx]

    best_weights, best_acc = run_genetic_algorithm(X_train, y_train, X_test, y_test)
    fold_accuracies.append(best_acc)

print(f"\nAverage 10-Fold CV Accuracy: {np.mean(fold_accuracies):.4f}")


--- Starting CV Fold 1 ---
Gen 1: Best Accuracy = 0.2292
Gen 2: Best Accuracy = 0.2292
Gen 3: Best Accuracy = 0.2292
Gen 4: Best Accuracy = 0.2292
Gen 5: Best Accuracy = 0.2292
Gen 6: Best Accuracy = 0.2292
Gen 7: Best Accuracy = 0.2292
Gen 8: Best Accuracy = 0.2292
Gen 9: Best Accuracy = 0.2292
Gen 10: Best Accuracy = 0.2292
Gen 11: Best Accuracy = 0.2292
Stopping early: No improvement in 10 generations.

--- Starting CV Fold 2 ---
Gen 1: Best Accuracy = 0.2000
Gen 2: Best Accuracy = 0.2000
Gen 3: Best Accuracy = 0.2000
Gen 4: Best Accuracy = 0.2105
Gen 5: Best Accuracy = 0.2105
Gen 6: Best Accuracy = 0.2105
Gen 7: Best Accuracy = 0.2316
Gen 8: Best Accuracy = 0.2316
Gen 9: Best Accuracy = 0.2421
Gen 10: Best Accuracy = 0.2526
Gen 11: Best Accuracy = 0.2632
Gen 12: Best Accuracy = 0.2737
Gen 13: Best Accuracy = 0.2737
Gen 14: Best Accuracy = 0.2737
Gen 15: Best Accuracy = 0.2737
Gen 16: Best Accuracy = 0.2737
Gen 17: Best Accuracy = 0.2737
Gen 18: Best Accuracy = 0.2737
Gen 19: Best 